### Importing Libraries

In [11]:
import torch
import tiktoken
import torch.nn as nn
from torch.nn import functional as F

### Loading the dataset

In [12]:
with open('/home/yon/Transformer/data/train.csv', 'r', encoding='utf-8') as f:
    text = f.read()


with open('/home/yon/Transformer/data/test.csv', 'r', encoding='utf-8') as f:
    text += f.read()

#### Explore the dataset

In [13]:
print("The length of the dataset is: ", len(text))
print(text[:1000])

The length of the dataset is:  1059640
text
"First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in h

## Tokenization Layer

In [14]:
encoding = tiktoken.get_encoding("o200k_base")

tokens = encoding.encode(text)
vocab_size = encoding.n_vocab
print(vocab_size)
print(len(tokens))
print(tokens[:10])

200019
282105
[919, 198, 1, 7127, 84479, 734, 13036, 581, 18988, 1062]


In [15]:
data = torch.tensor(tokens, dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([282105]) torch.int64
tensor([   919,    198,      1,   7127,  84479,    734,  13036,    581,  18988,
          1062,   6544,     11,   9598,    668,  10591,    364,   2594,    734,
        116872,     11,  10591,    364,   7127,  84479,    734,   3575,    553,
           722,  33944,   7542,    316,   1076,   1572,    316,   2079,   1109,
          1715,   2594,    734,  80773,     13,  33944,    364,   7127,  84479,
           734,   7127,     11,    481,   1761, 149492,    385,   3145, 137928,
           382,  20915,  20935,    316,    290,   1665,    364,   2594,    734,
          2167,   1761,   1507,     11,    581,   1761,   1507,    364,   7127,
         84479,    734,  12845,    765,  15874,   2395,     11,    326,  22782,
           679,  33994,    540,   1039,   2316,   3911,    558,   3031,   1507,
           261,  75722,   1715,   2594,    734,   3160,    945,  11695,    402,
          1507,     26,   1632,    480,    413,   4167,     25,   4194,     11,
       

#### Split up the data into train and validation sets

In [16]:
n = int(0.8*len(data)) # first 80% will be train, rest val
train_data = data[:n]
val_data = data[n:]

train_data[:10]

tensor([  919,   198,     1,  7127, 84479,   734, 13036,   581, 18988,  1062])

### Define Hyperparameters

In [17]:
batch_size = 16 
block_size = 32 
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

#### generate batch of data X and Y

In [18]:
def get_batch(split):
    data = train_data if split == 'train' else val_data

    ''' get a random number(index) and take a context
    window starting from there ( random context
    windows will help the model to learn better )'''

    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [19]:
x, y = get_batch('train')
print("when the input context window is like this: \n")
print(encoding.decode(x[1].tolist()))
print("\n\nthe out put will be like this:\n")
print(encoding.decode(y[1].tolist()))

when the input context window is like this: 

 would sustain some harm.

KING EDWARD IV:
Then get your husband's lands, to do them good.

LADY GREY:
Therefore I came unto your


the out put will be like this:

 sustain some harm.

KING EDWARD IV:
Then get your husband's lands, to do them good.

LADY GREY:
Therefore I came unto your maj


### Implementing a layer to calculate loss

In [20]:
@torch.no_grad()  # this stops the gradual descent ( it won't try to run gradual descent to update parameters )
def estimate_loss():
    out = {}
    model.eval()   # evaluation mode insures the model is not trying to learn( it stops dropout and BatchNorm )
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split) # select random batch to evaluate the model with
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

### Implement masking function

In [21]:
def mask(tensor, T, C):
  matrix = torch.zeros(1, T, C)
  mask = torch.triu(torch.ones_like(matrix[0, :, :]), diagonal=1)
  matrix.masked_fill_(mask.bool(), float('-inf'))
  return tensor + matrix



### Implementing a single attention head layer

In [22]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        # a neural layer which transforms the input vector into of dimension (B, T, C) -> (B, T, head_size)
        # key explains the tokens dentity(itself) / how it will be looked up by others
        self.key = nn.Linear(n_embd, head_size, bias=False)
        # query explains what the token is looking for
        self.query = nn.Linear(n_embd, head_size, bias=False)
        # value is the actual value of the token
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        ''' B -> Batch size(16)
            T -> Time(32) (Block Size) (context window size/ sequence length)
            C -> number of embedding for each token
        '''
        B,T,C = x.shape

        k = self.key(x)   # (B, T, C) -> (B, T, head_size)
        q = self.query(x) # (B, T, C) -> (B, T, head_size)

        # compute attention scores (how much token i matches token j)
        attention_score = q @ k.transpose(-2,-1) * C**-0.5
        attention_score = attention_score.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        attention_score = F.softmax(attention_score, dim=-1)
        attention_score = self.dropout(attention_score)

        # perform the weighted aggregation of the values
        v = self.value(x) # (B, T, C) -> (B, T, head_size)
        attention = attention_score @ v
        return attention

### Implement the multi Head Attention Layer

In [23]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # projects the result of each head into a linear layer
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
      # concatnate the result of each attention head
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # the result of each head is mixed
        out = self.dropout(self.projection(out))
        return out

### Implement a nural layer that introduces non-linearity

In [24]:
class FeedFoward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # linear layer which maps the input to each head (B, T, C) -> (B, T, c*4)
            # enables it to consider complex token features
            nn.Linear(n_embd, 4 * n_embd),
            # non-linear layer( activation function )
            nn.ReLU(),
            # linear layer which maps the head to input dimension (B, T, C*4) -> (B, T, C)
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## Transformer Layer

In [25]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads
        super().__init__()
        head_size = n_embd // n_head
        # calling multihead attention to get attention scores
        self.mHA = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        # apply normalization
        self.linearLayer1 = nn.LayerNorm(n_embd)
        self.linearLayer2 = nn.LayerNorm(n_embd)

    def forward(self, x):
      # residual connection
        x = x + self.mHA(self.linearLayer1(x))
        x = x + self.ffwd(self.linearLayer2(x))
        return x

### Decode Only Tokenizer

In [26]:
# super simple bigram model
class DecodOnlyTransformer(nn.Module):

    def __init__(self):
        super().__init__()
        # token embedding( Context wise )
        # projects vocab_size into n_embd dimension
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        # position embedding( Position wise )
        self.position_embedding = nn.Embedding(block_size, n_embd)

        # call the transformer block n_layer times
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.layernorm_f = nn.LayerNorm(n_embd)

        # projects the n_embd dimension to vocab_size dimension
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, token, targets=None):
        B, T = token.shape
        # token and targets are both (B,T) tensor of integers
        token_emb = self.token_embedding(token)
        position_emb = self.position_embedding(torch.arange(T, device=device)) # (T,C)
        x = token_emb + position_emb # (B,T,C)

        # this turns raw embeddings (x) into contextulized vector
        x = self.blocks(x) # (B,T,C)
        x = self.layernorm_f(x) # (B,T,C)

        # returns the prediction for the next token
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None: # if we are not training then the loss is not needed
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, token, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop token to the last block_size tokens
            token_cond = token[:, -block_size:]
            # get the predictions
            logits, loss = self(token_cond)
            # focus only on the last time step
            logits = logits[:, -1, :]
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            token_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled token to the running sequence
            token = torch.cat((token, token_next), dim=1) # (B, T+1)
        return token


## Train the model

In [27]:
model = DecodOnlyTransformer()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a 100 iteration evaluate the loss on train and val sets
    if iter % 100 == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # training and optimization
    # forward pass
    logits, loss = model(xb, yb)

    # backward pass
    optimizer.zero_grad(set_to_none=True)
    # calculates the gradients
    loss.backward()
    # optimizes the weights based on the calculated gradient
    optimizer.step()

26.003795 M parameters
step 0: train loss 12.3853, val loss 12.3873
step 100: train loss 7.1945, val loss 7.5960
step 200: train loss 7.0344, val loss 7.4823
step 300: train loss 6.6506, val loss 7.1770
step 400: train loss 6.2006, val loss 6.9030
step 500: train loss 5.9216, val loss 6.7510
step 600: train loss 5.7291, val loss 6.6227
step 700: train loss 5.5820, val loss 6.5750
step 800: train loss 5.4465, val loss 6.4704
step 900: train loss 5.3265, val loss 6.4747
step 1000: train loss 5.2256, val loss 6.4110
step 1100: train loss 5.1829, val loss 6.3354
step 1200: train loss 5.0688, val loss 6.3170
step 1300: train loss 5.0253, val loss 6.3503
step 1400: train loss 4.9838, val loss 6.2902
step 1500: train loss 4.9100, val loss 6.2251
step 1600: train loss 4.8797, val loss 6.2447


KeyboardInterrupt: 

### Save the model

In [ ]:
PATH = '/model/decoder_model.pth'

checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}


torch.save(checkpoint, PATH)

# Generate text

In [ ]:
def generate_text(prompt="The", max_new_tokens=200):

    checkpoint = torch.load(PATH, map_location=device)

    # Instantiate the model with the correct hyperparameters
    model = DecodOnlyTransformer()

    # Load the saved model weights
    model.load_state_dict(checkpoint['model_state_dict'])
    m = model.to(device)

    m.eval()
    print("Model loaded successfully.")

    # --- Generation ---

    # 1. ENCODE the start word (convert string to token IDs)
    print(f"Starting prompt: '{prompt}'")
    context_tokens = encoding.encode(prompt)

    # 2. CONVERT to a PyTorch tensor
    # The tensor must have shape (Batch Size, Sequence Length), so (1, T)
    context = torch.tensor([context_tokens], dtype=torch.long, device=device)

    # 3. GENERATE new token indices
    print("Generating text...")
    generated_indices = m.generate(context, max_new_tokens=max_new_tokens)[0].tolist()

    # 4. DECODE the full sequence (convert token IDs back to string)
    output_text = encoding.decode(generated_indices)

    return output_text

### model evaluation

In [ ]:
import math

def evaluate_model():
    """
    Evaluate the model and return perplexity on validation data
    """
    # Load the trained model
    checkpoint = torch.load(PATH, map_location=device)
    model = DecodOnlyTransformer()
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    print("Evaluating model on validation data...")
    
    total_loss = 0
    total_tokens = 0
    num_batches = 0
    
    with torch.no_grad():
        for _ in range(eval_iters):
            x, y = get_batch('val')
            
            # Forward pass
            logits, loss = model(x, y)
            
            if loss is not None:
                total_loss += loss.item() * x.numel()
                total_tokens += x.numel()
                num_batches += 1
    
    # Calculate average loss and perplexity
    if total_tokens > 0:
        avg_loss = total_loss / total_tokens
        perplexity = math.exp(avg_loss)
    else:
        avg_loss = float('inf')
        perplexity = float('inf')
    
    print(f"Evaluation completed:")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Perplexity: {perplexity:.4f}")
    
    return {
        'perplexity': perplexity,
        'loss': avg_loss,
        'note': f'Evaluated on {num_batches} batches, {total_tokens} tokens'
    }